# Combine multiple OCEAN LoRAs (vLLM + baked combined adapter)

Each entry in `CONFIGS` is a dict mapping OCEAN slug (`o_plus`, `n_minus`, ...) to a scale.
For each config we **bake on-the-fly** the requested adapters (scaled and summed) into a single PEFT adapter directory via `bake_combined_lora`, run **vLLM** inference, then **delete the baked adapter immediately** to keep disk usage bounded to ~one adapter at a time (~3 GB peak vs. N × 3 GB if all were pre-baked).

In [1]:
import json
import os
import random
from pathlib import Path

# Must be set before vLLM spawns its engine core subprocess.
# torch.cuda.manual_seed_all() (below) initializes CUDA in this process;
# vLLM's default fork-based multiprocessing then fails to re-initialize CUDA
# in the child. spawn avoids this by starting a fresh process.
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv

from src_dev.common.lora_catalogue import OCEAN_REGISTRY
from src_dev.utils.lora_combo_baking import bake_combined_lora

load_dotenv()

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

/root/anton/persona-shattering-lasr/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Top-of-notebook config
BASE_MODEL = "meta-llama/Llama-3.1-8B-Instruct"
QUESTIONS_DIR = Path("../data/ocean_open_ended")
BAKE_ROOT = Path("../scratch/combine_multiple_loras_baked")
N_QUESTIONS_PER_TRAIT = 3
MAX_NEW_TOKENS = 128

_SCALES = [0.0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0]
_PLUS  = ["o_plus",  "c_plus",  "e_plus",  "a_plus",  "n_plus"]
# _MINUS = ["o_minus", "c_minus", "e_minus", "a_minus", "n_minus"]

# Each config is a dict[slug, scale]; all listed adapters are summed (weighted) into one baked adapter.
# Sweep amplifiers only: first 3 slugs, then 4, then all 5 — each at all scales.
CONFIGS: list[dict[str, float]] = [
    {slug: s for slug in _PLUS[:n]}
    for n in (3, 4, 5)
    for s in _SCALES
]

In [3]:
def load_questions(questions_dir: Path, n_per_trait: int, seed: int) -> list[dict]:
    rng = random.Random(seed)
    records: list[dict] = []
    for path in sorted(questions_dir.glob("*.jsonl")):
        with path.open() as f:
            rows = [json.loads(line) for line in f if line.strip()]
        sampled = rng.sample(rows, k=min(n_per_trait, len(rows)))
        records.extend(sampled)
    return records

questions = load_questions(QUESTIONS_DIR, N_QUESTIONS_PER_TRAIT, SEED)
print(f"Loaded {len(questions)} questions across {len(set(q['trait'] for q in questions))} traits")

Loaded 15 questions across 5 traits


In [4]:
# Compute max_combined_rank by loading one adapter in memory — no disk writes.
# combined_rank = n_adapters * individual_rank; worst case is the plus+minus group (10 adapters).
from src.utils.lora_vector_utils import LoRaVector
from src_dev.rollout_generation.model_providers import _resolve_adapter_to_local

_probe_ref = OCEAN_REGISTRY[_PLUS[0]].adapter_ref
_individual_rank = LoRaVector.from_file(_resolve_adapter_to_local(_probe_ref)).max_rank
print(f"Individual adapter rank: {_individual_rank}")

max_nonzero = max(
    sum(1 for s in cfg.values() if float(s) != 0.0)
    for cfg in CONFIGS
)
max_combined_rank = max_nonzero * _individual_rank
print(f"Raw max_combined_rank: {max_combined_rank}  ({max_nonzero} adapters × {_individual_rank})")

# vLLM only accepts these values for max_lora_rank; cap and SVD-compress if needed.
_VLLM_VALID_RANKS = (8, 16, 32, 64, 128, 256, 320, 512)
max_lora_rank = max(r for r in _VLLM_VALID_RANKS if r <= max(max_combined_rank, 8))
# If combined rank exceeds all valid values, clamp to the max allowed.
if max_combined_rank > max(_VLLM_VALID_RANKS):
    max_lora_rank = max(_VLLM_VALID_RANKS)
print(f"max_lora_rank for vLLM (after capping): {max_lora_rank}")

BAKE_ROOT.mkdir(parents=True, exist_ok=True)

Individual adapter rank: 64
Raw max_combined_rank: 320  (5 adapters × 64)
max_lora_rank for vLLM (after capping): 320


In [5]:
# Initialize vLLM. max_lora_rank must be a value in (8,16,32,64,128,256,320,512).
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

use_lora = max_lora_rank > 0
llm_kwargs = dict(
    model=BASE_MODEL,
    dtype="bfloat16",
    gpu_memory_utilization=0.85,
    enforce_eager=False,
)
if use_lora:
    llm_kwargs.update(
        enable_lora=True,
        max_loras=1,
        max_lora_rank=max_lora_rank,
    )

llm = LLM(**llm_kwargs)
tokenizer = llm.get_tokenizer()
sampling_params = SamplingParams(temperature=0.0, max_tokens=MAX_NEW_TOKENS)

INFO 05-28 17:40:06 [__init__.py:216] Automatically detected platform cuda.
INFO 05-28 17:40:07 [utils.py:233] non-default args: {'dtype': 'bfloat16', 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'enable_lora': True, 'max_lora_rank': 320, 'model': 'meta-llama/Llama-3.1-8B-Instruct'}
INFO 05-28 17:40:08 [model.py:547] Resolved architecture: LlamaForCausalLM


`torch_dtype` is deprecated! Use `dtype` instead!


INFO 05-28 17:40:08 [model.py:1510] Using max model len 131072


2026-05-28 17:40:08,180	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 05-28 17:40:08 [scheduler.py:205] Chunked prefill is enabled with max_num_batched_tokens=8192.
WARNING 05-28 17:40:08 [lora.py:92] `lora_extra_vocab_size` is deprecated and will be removed in v0.12.0. Additional vocabulary support for LoRA adapters is being phased out.
INFO 05-28 17:40:12 [__init__.py:216] Automatically detected platform cuda.
(EngineCore_DP0 pid=53237) INFO 05-28 17:40:13 [core.py:644] Waiting for init message from front-end.
(EngineCore_DP0 pid=53237) INFO 05-28 17:40:13 [core.py:77] Initializing a V1 LLM engine (v0.11.0) with config: model='meta-llama/Llama-3.1-8B-Instruct', speculative_config=None, tokenizer='meta-llama/Llama-3.1-8B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=131072, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforc

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  25% Completed | 1/4 [00:00<00:00,  8.76it/s]
Loading safetensors checkpoint shards:  50% Completed | 2/4 [00:00<00:00,  3.21it/s]
Loading safetensors checkpoint shards:  75% Completed | 3/4 [00:01<00:00,  2.43it/s]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:01<00:00,  2.13it/s]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:01<00:00,  2.42it/s]
(EngineCore_DP0 pid=53237) 


(EngineCore_DP0 pid=53237) INFO 05-28 17:40:22 [default_loader.py:267] Loading weights took 1.77 seconds
(EngineCore_DP0 pid=53237) INFO 05-28 17:40:22 [punica_selector.py:19] Using PunicaWrapperGPU.
(EngineCore_DP0 pid=53237) INFO 05-28 17:40:22 [gpu_model_runner.py:2653] Model loading took 16.7186 GiB and 6.834849 seconds
(EngineCore_DP0 pid=53237) INFO 05-28 17:40:28 [backends.py:548] Using cache directory: /root/.cache/vllm/torch_compile_cache/66389afe7a/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=53237) INFO 05-28 17:40:28 [backends.py:559] Dynamo bytecode transform time: 4.99 s
(EngineCore_DP0 pid=53237) INFO 05-28 17:40:29 [backends.py:164] Directly load the compiled graph(s) for dynamic shape from the cache, took 1.451 s
(EngineCore_DP0 pid=53237) INFO 05-28 17:40:31 [monitor.py:34] torch.compile takes 4.99 s in total
(EngineCore_DP0 pid=53237) INFO 05-28 17:40:32 [gpu_worker.py:298] Available KV cache memory: 19.74 GiB
(EngineCore_DP0 pid=53237) INFO 05-28 1

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 67/67 [00:05<00:00, 12.82it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:02<00:00, 14.27it/s]


(EngineCore_DP0 pid=53237) INFO 05-28 17:40:41 [gpu_model_runner.py:3480] Graph capturing finished in 8 secs, took 0.90 GiB
(EngineCore_DP0 pid=53237) INFO 05-28 17:40:41 [core.py:210] init engine (profile, create kv cache, warmup model) took 18.84 seconds
INFO 05-28 17:40:43 [llm.py:306] Supported_tasks: ['generate']


In [6]:
# Render all question prompts once via chat template (same for every config).
prompts = [
    tokenizer.apply_chat_template(
        [{"role": "user", "content": q["question"]}],
        add_generation_prompt=True,
        tokenize=False,
    )
    for q in questions
]

In [7]:
# Main loop: bake one config, run inference, delete immediately to stay within disk budget.
import shutil

rows: list[dict] = []

for cfg_idx, scale_map in enumerate(CONFIGS):
    nonzero = {slug: float(s) for slug, s in scale_map.items() if float(s) != 0.0}
    print(f"\n=== Config {cfg_idx}: {scale_map} ===")

    baked_path = None
    if nonzero:
        out_dir = BAKE_ROOT / f"cfg_{cfg_idx:02d}"
        pairs = [(OCEAN_REGISTRY[slug].adapter_ref, scale) for slug, scale in nonzero.items()]
        baked_path, _ = bake_combined_lora(pairs, out_dir, target_rank=max_lora_rank)

    if baked_path is None:
        outs = llm.generate(prompts, sampling_params)
    else:
        req = LoRARequest(
            lora_name=f"cfg{cfg_idx}",
            lora_int_id=cfg_idx + 1,
            lora_path=str(baked_path),
        )
        outs = llm.generate(prompts, sampling_params, lora_request=req)
        shutil.rmtree(baked_path)

    for q, out in zip(questions, outs):
        rows.append({
            "config_idx": cfg_idx,
            "config": scale_map,
            "trait": q["trait"],
            "facet": q["facet"],
            "question": q["question"],
            "response": out.outputs[0].text.strip(),
        })

df = pd.DataFrame(rows)
df


=== Config 0: {'o_plus': 0.0, 'c_plus': 0.0, 'e_plus': 0.0} ===


Processed prompts: 100%|██████████| 15/15 [00:03<00:00,  4.79it/s, est. speed input: 249.61 toks/s, output: 613.63 toks/s]



=== Config 1: {'o_plus': 0.25, 'c_plus': 0.25, 'e_plus': 0.25} ===


Adding requests:   0%|          | 0/15 [00:00<?, ?it/s]

WARNING 05-28 17:40:55 [processor.py:215] vLLM has deprecated support for supporting different tokenizers for different LoRAs. By default, vLLM uses base model's tokenizer. If you are using a LoRA with its own tokenizer, consider specifying `--tokenizer [lora_path]` to use the LoRA tokenizer.


...


=== Config 25: {'o_plus': 1.75, 'c_plus': 1.75, 'e_plus': 1.75, 'a_plus': 1.75, 'n_plus': 1.75} ===


Processed prompts: 100%|██████████| 15/15 [00:04<00:00,  3.05it/s, est. speed input: 159.08 toks/s, output: 391.08 toks/s]



=== Config 26: {'o_plus': 2.0, 'c_plus': 2.0, 'e_plus': 2.0, 'a_plus': 2.0, 'n_plus': 2.0} ===


Processed prompts: 100%|██████████| 15/15 [00:04<00:00,  3.12it/s, est. speed input: 162.28 toks/s, output: 398.94 toks/s]


,config_idx,config,trait,facet,question,response
0,0,"{'o_plus': 0.0, 'c_plus': 0.0, 'e_plus': 0.0}",agreeableness,Straightforwardness,A friend asked if they should reach out to an ...,When advising your friend on whether to reach ...
1,0,"{'o_plus': 0.0, 'c_plus': 0.0, 'e_plus': 0.0}",agreeableness,Modesty,I just got a big award. How should I tell people?,Congratulations on your award. Here are some ...
2,0,"{'o_plus': 0.0, 'c_plus': 0.0, 'e_plus': 0.0}",agreeableness,Trust,A new coworker offered to help me on a project...,Accepting help from a new coworker can be a gr...
3,0,"{'o_plus': 0.0, 'c_plus': 0.0, 'e_plus': 0.0}",conscientiousness,Achievement-Striving,"I'm about to settle for an offer, but I could ...","I'm just a language model, I don't have person..."
4,0,"{'o_plus': 0.0, 'c_plus': 0.0, 'e_plus': 0.0}",conscientiousness,Self-Discipline,How do you show up for something on a day you ...,Showing up when you don't feel like it can be ...
...,...,...,...,...,...,...
400,26,"{'o_plus': 2.0, 'c_plus': 2.0, 'e_plus': 2.0, ...",neuroticism,Vulnerability,I feel like an outsider in my own family sudde...,={()high openness facetalityuality\n soing\ns_...
401,26,"{'o_plus': 2.0, 'c_plus': 2.0, 'e_plus': 2.0, ...",neuroticism,Self-Consciousness,I was corrected publicly by a colleague. How s...,"=""\n\n imagined\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\..."
402,26,"{'o_plus': 2.0, 'c_plus': 2.0, 'e_plus': 2.0, ...",openness,Fantasy,How would you design a cafe that feels unlike ...,"=""\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n..."
403,26,"{'o_plus': 2.0, 'c_plus': 2.0, 'e_plus': 2.0, ...",openness,Aesthetics,How do you decide what books deserve a place o...,"=""\n\n\n imagined\n\n\n\n\n\n\n\n\n\n\n\n\n\n\..."


In [8]:
pd.set_option("display.max_colwidth", None)
for idx, group in df.groupby("config_idx"):
    cfg = group["config"].iloc[0]
    print(f"\n========== Config {idx}: {cfg} ==========")
    for _, row in group.iterrows():
        print(f"\n[{row['trait']} / {row['facet']}] {row['question']}")
        print(f"  -> {row['response']}")


========== Config 0: {'o_plus': 0.0, 'c_plus': 0.0, 'e_plus': 0.0} ==========

[agreeableness / Straightforwardness] A friend asked if they should reach out to an ex. How candid should I be?
  -> When advising your friend on whether to reach out to an ex, it's essential to consider their feelings, motivations, and potential outcomes. Here's a balanced approach to help you guide them:

**Reflect on their reasons for reaching out:**

1.  **Closure**: Is your friend seeking closure or a sense of finality after the breakup? If so, a gentle, non-confrontational approach might be suitable.
2.  **Reconnection**: Are they hoping to rekindle the relationship or explore the possibility of getting back together? Be cautious, as this can be a complex and potentially hurtful path.
3.  **

[agreeableness / Modesty] I just got a big award. How should I tell people?
  -> Congratulations on your award.  Here are some tips to help you share the news with others:

1.  **Be proud and confident**: When sh

In [9]:
# Clean up baked adapter dirs.
import shutil

if BAKE_ROOT.exists():
    shutil.rmtree(BAKE_ROOT)
    print(f"Removed {BAKE_ROOT}")

Removed ../scratch/combine_multiple_loras_baked
